In [ ]:
# -----------------------------
# 1) Install required packages
# -----------------------------
!pip install -q transformers accelerate sentencepiece

In [ ]:
# -----------------------------
# 2) Imports
# -----------------------------
import os
import torch
import pandas as pd
import scipy.io.wavfile as wavfile
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from google.colab import drive

In [ ]:
# -----------------------------
# 3) Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -----------------------------
# 4) Define save directory
# -----------------------------
# This is where all generated .wav files will be stored
SAVE_DIR = "/content/drive/MyDrive/MusicGen_Project/Data/MusicGen_Output"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Files will be saved in: {SAVE_DIR}")


Files will be saved in: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output


In [ ]:
# -----------------------------
# 5) Load MusicGen model
# -----------------------------
#It can be switched to:
# "facebook/musicgen-small"
# "facebook/musicgen-medium"...
MODEL_NAME = "facebook/musicgen-small"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = MusicgenForConditionalGeneration.from_pretrained(MODEL_NAME)

# Use GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Using device:", device)
print("Loaded model:", MODEL_NAME)


In [ ]:
# -------------------------------------------------------
# 6) Define prompts: 6 categories, 5 prompts each = 30
# -------------------------------------------------------
prompt_dict = {
    "piano": [
        "a calm solo piano melody with soft emotional notes",
        "a bright classical piano piece with flowing arpeggios",
        "a melancholic piano composition with slow expressive phrasing",
        "an intimate jazz piano improvisation in a cozy lounge",
        "a cinematic grand piano theme with gentle dynamics"
    ],
    "guitar": [
        "a warm acoustic guitar fingerpicking pattern with a relaxed mood",
        "a soft classical guitar melody played with delicate expression",
        "an energetic strummed acoustic guitar rhythm for an upbeat song",
        "a bluesy electric guitar riff with smooth phrasing",
        "a dreamy clean electric guitar melody with light reverb"
    ],
    "drums": [
        "a tight acoustic drum groove with realistic kick snare and hi hats",
        "a punchy rock drum beat with strong snare hits",
        "a laid back jazz drum rhythm with brushes and subtle cymbals",
        "a fast electronic drum pattern with crisp percussion",
        "a funky drum groove with syncopated rhythm and dynamic fills"
    ],
    "ambient": [
        "a peaceful ambient soundscape with soft pads and airy textures",
        "a dark ambient atmosphere with deep drones and slow evolution",
        "an ethereal ambient track with shimmering textures and space",
        "a meditative ambient background with warm sustained tones",
        "a dreamy floating ambient composition with gentle motion"
    ],
    "electronic": [
        "an upbeat electronic dance groove with synth bass and clean drums",
        "a chill electronic track with soft synths and mellow rhythm",
        "a futuristic electronic piece with pulsing bass and layered textures",
        "a synthwave instrumental with retro 1980s electronic vibes",
        "a minimal electronic beat with deep bass and spacious effects"
    ],
    "orchestral": [
        "a cinematic orchestral theme with strings brass and percussion",
        "a dramatic orchestral soundtrack cue with rising tension",
        "a soft orchestral piece led by strings and woodwinds",
        "an epic trailer style orchestra with powerful drums and brass",
        "a romantic orchestral melody with lush strings and gentle piano"
    ]
}

In [ ]:
# -------------------------------------------------------
# 7) Convert prompt dictionary into a dataframe for order
# -------------------------------------------------------
rows = []
for category, prompts in prompt_dict.items():
    for i, prompt in enumerate(prompts, start=1):
        rows.append({
            "category": category,
            "prompt_id": i,
            "prompt_text": prompt
        })

prompts_df = pd.DataFrame(rows)

# Save prompt list as CSV for documentation / reproducibility
csv_path = os.path.join(SAVE_DIR, "prompts.csv")
prompts_df.to_csv(csv_path, index=False)

print(f"Prompt list saved to: {csv_path}")
print(prompts_df)


Prompt list saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/prompts.csv
      category  prompt_id                                        prompt_text
0        piano          1  a calm solo piano melody with soft emotional n...
1        piano          2  a bright classical piano piece with flowing ar...
2        piano          3  a melancholic piano composition with slow expr...
3        piano          4  an intimate jazz piano improvisation in a cozy...
4        piano          5  a cinematic grand piano theme with gentle dyna...
5       guitar          1  a warm acoustic guitar fingerpicking pattern w...
6       guitar          2  a soft classical guitar melody played with del...
7       guitar          3  an energetic strummed acoustic guitar rhythm f...
8       guitar          4  a bluesy electric guitar riff with smooth phra...
9       guitar          5  a dreamy clean electric guitar melody with lig...
10       drums          1  a tight acoustic drum groove with re

In [ ]:
# -------------------------------------------------------
# 8) Generation settings
# -------------------------------------------------------
# max_new_tokens controls duration.
# Around 256 tokens is commonly used for short clips (~5 sec range).
MAX_NEW_TOKENS = 256
GUIDANCE_SCALE = 3.0
DO_SAMPLE = True

In [ ]:
# -------------------------------------------------------
# 9) Generate and save each audio file
# -------------------------------------------------------
sampling_rate = model.config.audio_encoder.sampling_rate

for idx, row in prompts_df.iterrows():
    category = row["category"]
    prompt_id = row["prompt_id"]
    prompt_text = row["prompt_text"]

    print("=" * 70)
    print(f"Generating [{idx+1}/{len(prompts_df)}]")
    print(f"Category : {category}")
    print(f"Prompt ID: {prompt_id}")
    print(f"Prompt   : {prompt_text}")

    # Prepare text input
    inputs = processor(
        text=[prompt_text],
        padding=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Generate audio
    with torch.no_grad():
        audio_values = model.generate(
            **inputs,
            do_sample=DO_SAMPLE,
            guidance_scale=GUIDANCE_SCALE,
            max_new_tokens=MAX_NEW_TOKENS
        )

    # Extract waveform from tensor
    audio = audio_values[0, 0].detach().cpu().numpy()

    # Convert float audio in [-1, 1] to int16 for WAV saving
    audio_int16 = (audio * 32767).astype("int16")

    # Create filename
    # Example: piano_01.wav, electronic_03.wav, etc.
    filename = f"{category}_{prompt_id:02d}.wav"
    filepath = os.path.join(SAVE_DIR, filename)

    # Save WAV file
    wavfile.write(filepath, sampling_rate, audio_int16)

    print(f"Saved to: {filepath}")

print("\nAll 30 audio files have been generated and saved successfully.")
print(f"Output folder: {SAVE_DIR}")

Generating [1/30]
Category : piano
Prompt ID: 1
Prompt   : a calm solo piano melody with soft emotional notes
Saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/piano_01.wav
Generating [2/30]
Category : piano
Prompt ID: 2
Prompt   : a bright classical piano piece with flowing arpeggios
Saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/piano_02.wav
Generating [3/30]
Category : piano
Prompt ID: 3
Prompt   : a melancholic piano composition with slow expressive phrasing
Saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/piano_03.wav
Generating [4/30]
Category : piano
Prompt ID: 4
Prompt   : an intimate jazz piano improvisation in a cozy lounge
Saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/piano_04.wav
Generating [5/30]
Category : piano
Prompt ID: 5
Prompt   : a cinematic grand piano theme with gentle dynamics
Saved to: /content/drive/MyDrive/MusicGen_Project/MusicGen_Output/piano_05.wav
Generating [6/30]
Category : guitar
Pr